In [0]:
%sql
create table if not exists instacart.gold_schema.detailed_order_t as 
select o.order_id, o.user_id, o.eval_set, o.order_number, o.order_dow, o.order_hour_of_day, o.days_since_prior_order, p.product_id, p.product_name,
a.aisle_id, a.aisle_name, d.department_id, d.department, op.add_to_cart_order, op.reordered
 from instacart.silver_schema.product_t p, instacart.silver_schema.orders_t o, 
 instacart.silver_schema.aisles_t a, instacart.silver_schema.departments_t d, instacart.silver_schema.order_product_t op 
where p.product_id = op.product_id and o.order_id = op.order_id and a.aisle_id = p.aisle_id and d.department_id = p.department_id;

In [0]:
select count(*) from instacart.gold_schema.detailed_order_t
where product_id in (47209,13176)


categories from which people reorder

In [0]:
create table if not exists instacart.gold_schema.department_orders_t as
select do.department,count(*) as total_order,sum(CAST(reordered AS INT)) as reorder, round(((reorder/count(*))*100),2) as reorder_rate_percentage
from instacart.gold_schema.detailed_order_t do
group by do.department
order by total_order desc;

In [0]:
select * from instacart.gold_schema.department_orders_t

most popular product

In [0]:
create temporary view popular_product as
select department,aisle_name,product_name, count(*) as total_orders, sum(CAST(reordered AS INT)) as reorder 
from instacart.gold_schema.detailed_order_t 
group by product_name,department,aisle_name
order by total_orders desc;

In [0]:
select * from popular_product;

Order timing pattern

In [0]:
create view order_time as
select 
case 
 when order_dow =1 then 'Monday'
 when order_dow =2 then 'Tuesday'
 when order_dow =3 then 'Wednesday'
 when order_dow =4 then 'Thursday'
 when order_dow =5 then 'Friday'
 when order_dow =6 then 'Saturday'
 when order_dow =0 then 'Sunday'
end as order_dow, order_hour_of_day,count(*)
from instacart.gold_schema.detailed_order_t
group by order_dow, order_hour_of_day
order by count(*) desc

In [0]:
select * from order_time

loyal customer/ frequent buyer

In [0]:
create view loyal_cust as
select user_id,count(distinct(order_id)) as total_order,count(*) as item_purchased, sum(int(reordered)) as reordered
from instacart.gold_schema.detailed_order_t
group by user_id
order by total_order 

which products are bought together

In [0]:
create view coupled_products as(
WITH ss AS (
    SELECT order_id,product_id
    FROM instacart.gold_schema.detailed_order_t
)
SELECT do.product_id as product_id_1, ss.product_id as product_id_2, count(*) as pair_count
FROM instacart.gold_schema.detailed_order_t do, ss
where ss.order_id = do.order_id 
and ss.product_id < do.product_id
GROUP BY do.product_id,ss.product_id
having count(*) > 1
ORDER BY count(*) desc)

In [0]:
select * from coupled_products;